# Cuatro conceptos de fine-tuning, cuatro demos

| Concepto | Qué se muestra en este notebook |
| --- | --- |
| **Transfer Learning** | BETO (BERT en español) convertido en clasificador de sentimientos, comparado contra el mismo modelo entrenado desde cero |
| **Supervised Fine-Tuning** | Un LLM aprende a extraer datos de un mensaje de inscripción y entregarlos en un JSON con formato fijo |
| **Instruction Tuning** | Un modelo *base* (que solo completa texto) aprende a seguir instrucciones al estilo Alpaca |
| **PEFT** | Cuántos parámetros se entrenan y cuánta memoria se ahorra con LoRA frente a ajustar el modelo completo |

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU T4**.

Cada parte se puede ejecutar por separado después de la celda de configuración. Al final de cada parte se libera la GPU.

## 0. Configuración común

In [ ]:
!pip install -q peft datasets accelerate transformers

In [ ]:
import gc, json, os, random
import torch
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification,
                          AutoConfig, Trainer, TrainingArguments, set_seed)
from peft import LoraConfig, get_peft_model

assert torch.cuda.is_available(), "Activa la GPU: Entorno de ejecución > Cambiar tipo > T4"
set_seed(42)

def liberar_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def entrenar_lora(model, tokenizer, textos, epocas=5, r=16, lr=2e-4, carpeta="salida"):
    """Envuelve el modelo con LoRA y lo entrena con una lista de textos ya formateados."""
    cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, task_type="CAUSAL_LM",
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                     "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()

    ds = Dataset.from_dict({"texto": textos}).map(
        lambda e: tokenizer(e["texto"], truncation=True, max_length=384), remove_columns=["texto"])

    def collator(batch):
        # Solo se ignora el relleno (padding). Así el modelo también aprende a emitir el token de fin.
        lote = tokenizer.pad(batch, return_tensors="pt")
        etiquetas = lote["input_ids"].clone()
        etiquetas[lote["attention_mask"] == 0] = -100
        lote["labels"] = etiquetas
        return lote

    args = TrainingArguments(output_dir=carpeta, num_train_epochs=epocas, per_device_train_batch_size=8,
                             learning_rate=lr, lr_scheduler_type="cosine", warmup_steps=3,
                             logging_steps=5, fp16=True, save_strategy="no", report_to="none",
                             remove_unused_columns=False)
    trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)
    trainer.train()
    model.eval()
    return model, trainer

---
# Parte 1. Transfer Learning: BETO → clasificador de sentimientos

**Idea:** un modelo que ya leyó millones de textos en español "sabe" mucho del idioma. Reutilizar esos pesos permite aprender una tarea nueva con muy pocos ejemplos.

**Experimento:** se entrena el *mismo* modelo con los *mismos* 40 ejemplos de dos formas:
1. Partiendo de los pesos preentrenados de BETO (transfer learning).
2. Partiendo de pesos aleatorios (desde cero).

Además se mide BETO sin entrenar, como línea base. El conjunto de prueba usa palabras que **no** aparecen en el entrenamiento ("maravillosa", "espantosa", "mediocre").

BETO fue creado por el Departamento de Ciencias de la Computación de la Universidad de Chile.

In [ ]:
pos_train = [
    "Me encantó el taller, aprendí muchísimo.", "La profesora explica súper claro.",
    "Excelente curso, lo recomiendo.", "Quedé muy contenta con la clase de hoy.",
    "El material está bien hecho y es útil.", "Por fin entendí cómo usar Excel.",
    "Muy buena atención, respondieron todas mis dudas.", "La plataforma funciona rápido y es fácil de usar.",
    "Fue una experiencia muy bonita.", "Los ejemplos eran entretenidos y prácticos.",
    "Me sirvió harto para mi emprendimiento.", "Gracias, el curso superó mis expectativas.",
    "Las clases son dinámicas y motivadoras.", "Aprendí a usar la inteligencia artificial en mi negocio.",
    "Todo muy ordenado y puntual.", "El grupo era muy acogedor.",
    "Recomiendo este programa a todas mis amigas.", "La certificación me abrió nuevas oportunidades.",
    "Me gustó mucho la actividad práctica.", "Buen contenido y buena onda.",
]
neg_train = [
    "No entendí nada de la clase.", "El audio se escuchaba pésimo.",
    "Muy aburrido, me quedé dormido.", "La plataforma se cae a cada rato.",
    "Nadie respondió mis preguntas.", "El curso fue una pérdida de tiempo.",
    "Los videos son muy largos y confusos.", "Estoy decepcionada con el taller.",
    "La clase empezó tarde y terminó a la rápida.", "El material tiene muchos errores.",
    "No me sirvió para nada.", "Me costó mucho entrar a la sesión y nadie ayudó.",
    "Malísima experiencia, no lo recomiendo.", "El profesor iba demasiado rápido.",
    "Las instrucciones de la tarea eran confusas.", "Me sentí perdida todo el tiempo.",
    "La conexión era horrible.", "Prometieron certificado y nunca llegó.",
    "Demasiada teoría y nada práctico.", "Me arrepiento de haberme inscrito.",
]
pos_test = [
    "Una clase maravillosa.", "Quedé fascinado con lo que aprendimos.", "Espectacular, volvería a tomarlo.",
    "La relatora fue increíble.", "Súper recomendable para principiantes.", "Disfruté cada sesión.",
    "Contenido valioso y bien explicado.", "Estoy feliz con mi progreso.",
    "Una iniciativa genial para la comunidad.", "Salí muy motivada a seguir aprendiendo.",
]
neg_test = [
    "Fue terrible.", "Qué desastre de organización.", "Una clase espantosa.", "Me frustré muchísimo.",
    "No volvería nunca.", "Pésima experiencia de principio a fin.", "Todo fue lento y desordenado.",
    "Me pareció inútil.", "Estoy muy molesta con el servicio.", "Un curso mediocre.",
]
X_train, y_train = pos_train + neg_train, [1] * 20 + [0] * 20
X_test,  y_test  = pos_test + neg_test,   [1] * 10 + [0] * 10
ETIQUETAS = {0: "negativo 😞", 1: "positivo 😀"}
print(f"{len(X_train)} ejemplos de entrenamiento, {len(X_test)} de prueba")

In [ ]:
BERT = "dccuchile/bert-base-spanish-wwm-cased"
tok_bert = AutoTokenizer.from_pretrained(BERT)

def evaluar_clf(modelo):
    modelo.eval()
    enc = tok_bert(X_test, padding=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        pred = modelo(**enc).logits.argmax(-1).tolist()
    return sum(p == y for p, y in zip(pred, y_test)) / len(y_test)

def entrenar_clf(modelo, epocas=8, lr=3e-5):
    opt = torch.optim.AdamW(modelo.parameters(), lr=lr)
    enc = tok_bert(X_train, padding=True, return_tensors="pt")
    y = torch.tensor(y_train)
    for _ in range(epocas):
        modelo.train()
        perm = torch.randperm(len(y))
        for i in range(0, len(y), 8):
            idx = perm[i:i + 8]
            salida = modelo(input_ids=enc["input_ids"][idx].cuda(),
                            attention_mask=enc["attention_mask"][idx].cuda(),
                            labels=y[idx].cuda())
            salida.loss.backward()
            opt.step()
            opt.zero_grad()
    return modelo

resultados = {}

# a) BETO preentrenado, con la "cabeza" de clasificación nueva y sin entrenar
set_seed(42)
beto = AutoModelForSequenceClassification.from_pretrained(BERT, num_labels=2).cuda()
resultados["BETO sin ajustar"] = evaluar_clf(beto)

# b) Transfer learning: BETO preentrenado + ajuste con 40 ejemplos
beto = entrenar_clf(beto)
resultados["BETO + 40 ejemplos\n(transfer learning)"] = evaluar_clf(beto)

# c) Misma arquitectura, pesos aleatorios, mismos 40 ejemplos
set_seed(42)
config = AutoConfig.from_pretrained(BERT, num_labels=2)
desde_cero = AutoModelForSequenceClassification.from_config(config).cuda()
desde_cero = entrenar_clf(desde_cero)
resultados["Desde cero + 40 ejemplos"] = evaluar_clf(desde_cero)

for k, v in resultados.items():
    print(f"{k.replace(chr(10), ' '):45s} precisión: {v:.0%}")

In [ ]:
plt.figure(figsize=(8, 4))
barras = plt.bar(list(resultados.keys()), list(resultados.values()), color=["#bbbbbb", "#2e7d32", "#c62828"])
plt.axhline(0.5, ls="--", color="gray", label="Adivinar al azar")
plt.bar_label(barras, labels=[f"{v:.0%}" for v in resultados.values()])
plt.ylim(0, 1.1); plt.ylabel("Precisión en prueba"); plt.legend()
plt.title("Mismos datos, mismo modelo: lo que cambia es el punto de partida")
plt.show()

### Prueba en vivo

In [ ]:
def sentimiento(texto):
    beto.eval()
    enc = tok_bert(texto, return_tensors="pt").to("cuda")
    with torch.no_grad():
        probs = beto(**enc).logits.softmax(-1)[0]
    clase = int(probs.argmax())
    return f"{ETIQUETAS[clase]}  ({probs[clase]:.0%} de confianza)"

for frase in ["El taller estuvo bacán", "La clase fue una lata", "No estuvo mal, pero podría mejorar"]:
    print(f"{frase:40s} → {sentimiento(frase)}")

In [ ]:
del beto, desde_cero
liberar_gpu()

---
# Parte 2. Supervised Fine-Tuning: de mensaje libre a JSON

**Idea:** se le muestran al modelo pares de entrada y salida etiquetados para una tarea **estructurada**. Aquí la tarea es leer mensajes de inscripción escritos de cualquier forma y devolver siempre el mismo JSON:

`{"nombre": ..., "edad": ..., "comuna": ..., "curso": ...}` (con `null` si falta un dato).

La métrica es objetiva: ¿el JSON es válido? ¿Los cuatro campos son exactamente correctos?

Los mensajes de prueba usan **redacciones distintas** a las de entrenamiento. Los nombres son ficticios.

In [ ]:
random.seed(0)
NOMBRES = ["María José Soto", "Pedro Huenchullán", "Carmen Rojas", "Luis Pérez", "Ana Millán",
           "Jorge Cárcamo", "Rosa Painequeo", "Felipe Araya", "Daniela Muñoz", "Héctor Vidal",
           "Valentina Tapia", "Juan Carlos Oyarzún"]
COMUNAS = ["Castro", "Ancud", "Maipú", "Peñalolén", "Temuco", "Valparaíso", "Puerto Montt", "La Florida"]
CURSOS = ["Excel básico", "Marketing digital", "IA para emprendedoras",
          "Programación en Python", "Ventas por redes sociales"]

# (plantilla, ¿incluye edad?, ¿incluye comuna?)
PLANTILLAS_TRAIN = [
    ("Hola, soy {nombre}, tengo {edad} años, vivo en {comuna} y quiero inscribirme en {curso}.", True, True),
    ("Buenas tardes. Me llamo {nombre}, soy de {comuna}. Me interesa el curso de {curso}. Edad: {edad}.", True, True),
    ("{nombre} aquí 👋 desde {comuna}, quiero entrar a {curso}", False, True),
    ("Quisiera postular a {curso}. Mi nombre es {nombre} y tengo {edad}.", True, False),
    ("Inscripción: {nombre}, {edad} años, comuna de {comuna}, curso {curso}.", True, True),
]
PLANTILLAS_TEST = [
    ("Estimados, les escribe {nombre} ({edad} años), vecina de {comuna}. ¿Quedan cupos en {curso}?", True, True),
    ("Me gustaría tomar {curso}, soy {nombre}", False, False),
]

def crear_ejemplo(plantilla, con_edad, con_comuna):
    d = {"nombre": random.choice(NOMBRES), "edad": random.randint(18, 75),
         "comuna": random.choice(COMUNAS), "curso": random.choice(CURSOS)}
    mensaje = plantilla.format(**d)
    gold = {"nombre": d["nombre"], "edad": d["edad"] if con_edad else None,
            "comuna": d["comuna"] if con_comuna else None, "curso": d["curso"]}
    return mensaje, gold

train_sft = [crear_ejemplo(*random.choice(PLANTILLAS_TRAIN)) for _ in range(40)]
test_sft  = [crear_ejemplo(*PLANTILLAS_TEST[i % 2]) for i in range(10)]

print("Ejemplo de entrenamiento:\n ", train_sft[0][0], "\n →", json.dumps(train_sft[0][1], ensure_ascii=False))
print("\nEjemplo de prueba:\n ", test_sft[1][0], "\n →", json.dumps(test_sft[1][1], ensure_ascii=False))

In [ ]:
LLM_INSTRUCT = "Qwen/Qwen2.5-0.5B-Instruct"
tok_q = AutoTokenizer.from_pretrained(LLM_INSTRUCT)
modelo_sft = AutoModelForCausalLM.from_pretrained(LLM_INSTRUCT, torch_dtype=torch.float32).cuda()

SISTEMA_SFT = "Extrae los datos de inscripción del mensaje."

def chat(modelo, sistema, usuario, max_tokens=100):
    msgs = [{"role": "system", "content": sistema}, {"role": "user", "content": usuario}]
    prompt = tok_q.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok_q(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = modelo.generate(**ids, max_new_tokens=max_tokens, do_sample=False,
                              pad_token_id=tok_q.pad_token_id)
    return tok_q.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def evaluar_sft(modelo, mostrar=3):
    validos = correctos = 0
    for i, (mensaje, gold) in enumerate(test_sft):
        salida = chat(modelo, SISTEMA_SFT, mensaje)
        try:
            pred = json.loads(salida)
            validos += 1
            correctos += int(pred == gold)
        except json.JSONDecodeError:
            pass
        if i < mostrar:
            print(f"📩 {mensaje}\n🤖 {salida}\n" + "-" * 70)
    print(f"\nJSON válido: {validos}/{len(test_sft)}   ·   Exactamente correcto: {correctos}/{len(test_sft)}")
    return validos, correctos

print("=== ANTES del fine-tuning ===\n")
antes_sft = evaluar_sft(modelo_sft)

In [ ]:
def formatear_sft(mensaje, gold):
    msgs = [{"role": "system", "content": SISTEMA_SFT},
            {"role": "user", "content": mensaje},
            {"role": "assistant", "content": json.dumps(gold, ensure_ascii=False)}]
    return tok_q.apply_chat_template(msgs, tokenize=False)

textos_sft = [formatear_sft(m, g) for m, g in train_sft]
modelo_sft, _ = entrenar_lora(modelo_sft, tok_q, textos_sft, epocas=5, carpeta="sft")

In [ ]:
print("=== DESPUÉS del fine-tuning ===\n")
despues_sft = evaluar_sft(modelo_sft)

etiquetas = ["JSON válido", "Exactamente correcto"]
x = range(len(etiquetas))
plt.figure(figsize=(6, 4))
plt.bar([i - 0.2 for i in x], antes_sft, width=0.4, label="Antes", color="#bbbbbb")
plt.bar([i + 0.2 for i in x], despues_sft, width=0.4, label="Después", color="#1565c0")
plt.xticks(list(x), etiquetas); plt.ylim(0, len(test_sft) + 1); plt.ylabel(f"de {len(test_sft)} mensajes")
plt.title("Supervised Fine-Tuning con 40 ejemplos"); plt.legend(); plt.show()

In [ ]:
modelo_sft.save_pretrained("adaptador_sft")
del modelo_sft
liberar_gpu()

---
# Parte 3. Instruction Tuning: enseñar a un modelo base a seguir instrucciones

**Idea:** un modelo *base* solo aprendió a predecir la siguiente palabra. Si se le da una instrucción, tiende a seguir escribiendo más texto parecido en vez de responder. El instruction tuning (como Flan-T5 o Alpaca) lo entrena con **muchas instrucciones distintas** y sus respuestas, para que aprenda el patrón general "instrucción → respuesta → parar".

Se usa **Qwen2.5-0.5B base** (no la versión Instruct) y el formato de Alpaca. El dataset mezcla 32 ejemplos de 10 tipos de instrucción: resumir, traducir, clasificar, extraer palabras clave, responder, listar, corregir, titular, antónimos y saludos formales.

In [ ]:
instrucciones = [
    ("Resume el texto en una frase.", "El taller de marketing digital se realizó el sábado en Castro. Participaron 25 emprendedoras que aprendieron a crear publicaciones para Instagram y a medir sus resultados.", "25 emprendedoras de Castro aprendieron a crear y medir publicaciones en Instagram."),
    ("Resume el texto en una frase.", "La biblioteca municipal abrirá un nuevo horario los fines de semana. Desde marzo atenderá sábados y domingos de 10 a 14 horas para facilitar el acceso a estudiantes que trabajan.", "La biblioteca atenderá los fines de semana desde marzo para apoyar a estudiantes que trabajan."),
    ("Resume el texto en una frase.", "Un grupo de adultos mayores aprendió a usar videollamadas para hablar con sus nietos. Las clases duraron cuatro semanas y todos lograron conectarse solos al final.", "Adultos mayores aprendieron en cuatro semanas a hacer videollamadas por su cuenta."),
    ("Traduce al inglés.", "Me gusta aprender programación.", "I like learning programming."),
    ("Traduce al inglés.", "La clase empieza a las nueve.", "The class starts at nine."),
    ("Traduce al inglés.", "Necesito ayuda con mi computador.", "I need help with my computer."),
    ("Traduce al inglés.", "Mi emprendimiento vende pan amasado.", "My business sells homemade bread."),
    ("Clasifica el sentimiento como positivo o negativo.", "El curso fue excelente.", "positivo"),
    ("Clasifica el sentimiento como positivo o negativo.", "No me gustó la atención.", "negativo"),
    ("Clasifica el sentimiento como positivo o negativo.", "Aprendí muchísimo, gracias.", "positivo"),
    ("Clasifica el sentimiento como positivo o negativo.", "La aplicación se cae todo el tiempo.", "negativo"),
    ("Extrae tres palabras clave.", "La inteligencia artificial puede ayudar a pequeñas empresas a responder clientes y organizar inventario.", "inteligencia artificial, pequeñas empresas, inventario"),
    ("Extrae tres palabras clave.", "Las mujeres rurales participaron en un programa de alfabetización digital con celulares.", "mujeres rurales, alfabetización digital, celulares"),
    ("Extrae tres palabras clave.", "El volcán Osorno es uno de los destinos turísticos más visitados del sur de Chile.", "volcán Osorno, turismo, sur de Chile"),
    ("Responde la pregunta.", "¿Cuál es la capital de Chile?", "Santiago."),
    ("Responde la pregunta.", "¿Cuántos días tiene una semana?", "Siete."),
    ("Responde la pregunta.", "¿Qué color se obtiene al mezclar azul y amarillo?", "Verde."),
    ("Responde la pregunta.", "¿Para qué sirve una contraseña?", "Para proteger el acceso a una cuenta o dispositivo."),
    ("Convierte el texto en una lista con guiones.", "Para la clase necesitas cuaderno, lápiz y un celular con internet.", "- Cuaderno\n- Lápiz\n- Celular con internet"),
    ("Convierte el texto en una lista con guiones.", "Compra harina, levadura, sal y agua.", "- Harina\n- Levadura\n- Sal\n- Agua"),
    ("Convierte el texto en una lista con guiones.", "Los pasos son registrarse, ver el video y subir la tarea.", "- Registrarse\n- Ver el video\n- Subir la tarea"),
    ("Corrige la ortografía.", "mañana boy a la clace de exel", "Mañana voy a la clase de Excel."),
    ("Corrige la ortografía.", "ola, kiero inscribirme en el curzo", "Hola, quiero inscribirme en el curso."),
    ("Corrige la ortografía.", "la reunion es el lunes a las tres", "La reunión es el lunes a las tres."),
    ("Escribe un título breve para el texto.", "Vecinas de Ancud crearon una tienda en línea para vender sus tejidos a todo Chile.", "Tejedoras de Ancud venden en línea"),
    ("Escribe un título breve para el texto.", "Estudiantes de liceo construyeron un robot que separa basura reciclable.", "Robot escolar para reciclar"),
    ("Escribe un título breve para el texto.", "Una feria de ciencia reunió a más de mil familias en Temuco.", "Mil familias en feria de ciencia"),
    ("Escribe lo contrario de la palabra.", "rápido", "lento"),
    ("Escribe lo contrario de la palabra.", "abrir", "cerrar"),
    ("Escribe lo contrario de la palabra.", "feliz", "triste"),
    ("Escribe un saludo formal para un correo.", "Destinatario: señora Pérez", "Estimada señora Pérez:"),
    ("Escribe un saludo formal para un correo.", "Destinatario: equipo de soporte", "Estimado equipo de soporte:"),
]

prueba_inst = [
    ("Resume el texto en una frase.", "El municipio instaló puntos de wifi gratuito en diez plazas. La medida busca que más vecinos puedan hacer trámites en línea sin gastar sus datos móviles."),
    ("Traduce al inglés.", "El taller fue muy entretenido."),
    ("Convierte el texto en una lista con guiones.", "Trae tu carnet, un pendrive y tu cargador."),
    ("Corrige la ortografía.", "boy a llegar tarde a la clace"),
    ("Escribe un título breve para el texto.", "Un grupo de pescadores de Quellón aprendió a usar redes sociales para vender sus productos."),
]

def prompt_alpaca(instruccion, entrada):
    return f"### Instrucción:\n{instruccion}\n\n### Entrada:\n{entrada}\n\n### Respuesta:\n"

print(f"{len(instrucciones)} ejemplos, {len(set(i for i, _, _ in instrucciones))} tipos de instrucción")

In [ ]:
LLM_BASE = "Qwen/Qwen2.5-0.5B"   # versión base, SIN instruction tuning
tok_b = AutoTokenizer.from_pretrained(LLM_BASE)
modelo_base = AutoModelForCausalLM.from_pretrained(LLM_BASE, torch_dtype=torch.float32).cuda()

def completar(modelo, instruccion, entrada, max_tokens=80):
    ids = tok_b(prompt_alpaca(instruccion, entrada), return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = modelo.generate(**ids, max_new_tokens=max_tokens, do_sample=False,
                              repetition_penalty=1.1, eos_token_id=tok_b.eos_token_id,
                              pad_token_id=tok_b.eos_token_id)
    return tok_b.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

respuestas_antes_inst = {}
print("=== ANTES: modelo base ===\n")
for inst, ent in prueba_inst:
    respuestas_antes_inst[(inst, ent)] = completar(modelo_base, inst, ent)
    print(f"📝 {inst}  |  {ent}\n🤖 {respuestas_antes_inst[(inst, ent)]}\n" + "-" * 70)

In [ ]:
textos_inst = [prompt_alpaca(i, e) + s + tok_b.eos_token for i, e, s in instrucciones]
modelo_base, _ = entrenar_lora(modelo_base, tok_b, textos_inst, epocas=6, carpeta="inst")

In [ ]:
print("=== Comparación ANTES / DESPUÉS ===\n")
for inst, ent in prueba_inst:
    print(f"📝 {inst}  |  {ent}")
    print(f"\n🔵 ANTES:\n{respuestas_antes_inst[(inst, ent)]}")
    print(f"\n🟢 DESPUÉS:\n{completar(modelo_base, inst, ent)}")
    print("=" * 70)

### Prueba en vivo: una instrucción que no estaba en el entrenamiento
Si el modelo aprendió el *patrón* de seguir instrucciones, debería intentarlo también con tipos de tarea nuevos.

In [ ]:
print(completar(modelo_base, "Escribe una pregunta sobre el texto.",
                "El curso de Python dura ocho semanas y tiene clases los martes y jueves."))

In [ ]:
modelo_base.save_pretrained("adaptador_instrucciones")
del modelo_base
liberar_gpu()

---
# Parte 4. PEFT: ¿cuánto se ahorra con LoRA?

En las partes 2 y 3 se usó LoRA, una técnica de **Parameter-Efficient Fine-Tuning**: el modelo original queda congelado y solo se entrenan matrices pequeñas que se agregan a algunas capas.

Aquí se calcula, para Qwen2.5-0.5B, cuántos parámetros se entrenan y cuánta memoria de GPU requiere el entrenamiento en cada caso. La estimación usa una regla común: con el optimizador Adam en fp32, cada parámetro **entrenable** cuesta unos 16 bytes (peso, gradiente y dos estados del optimizador), y cada parámetro **congelado** unos 4 bytes. No incluye las activaciones.

In [ ]:
import pandas as pd

modelo_cpu = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", torch_dtype=torch.float32)
total = sum(p.numel() for p in modelo_cpu.parameters())
OBJETIVOS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

def params_lora(r):
    # LoRA agrega dos matrices por capa: A (entrada × r) y B (r × salida)
    return sum(r * (m.in_features + m.out_features)
               for nombre, m in modelo_cpu.named_modules()
               if isinstance(m, torch.nn.Linear) and nombre.split(".")[-1] in OBJETIVOS)

filas = [{"Método": "Fine-tuning completo", "Parámetros entrenables": total,
          "% del modelo": 100.0, "Memoria aprox. (GB)": total * 16 / 1e9}]
for r in [4, 16, 64]:
    n = params_lora(r)
    filas.append({"Método": f"LoRA r={r}", "Parámetros entrenables": n,
                  "% del modelo": 100 * n / total,
                  "Memoria aprox. (GB)": (total * 4 + n * 16) / 1e9})

tabla = pd.DataFrame(filas)
tabla["Parámetros entrenables"] = tabla["Parámetros entrenables"].map(lambda x: f"{x/1e6:,.1f} M")
tabla["% del modelo"] = tabla["% del modelo"].map(lambda x: f"{x:.2f} %")
tabla["Memoria aprox. (GB)"] = tabla["Memoria aprox. (GB)"].map(lambda x: f"{x:.1f}")
display(tabla)
del modelo_cpu

In [ ]:
def tamano_mb(carpeta):
    if not os.path.isdir(carpeta):
        return None
    return sum(os.path.getsize(os.path.join(carpeta, f)) for f in os.listdir(carpeta)) / 1e6

modelo_mb = total * 2 / 1e6   # el modelo completo guardado en fp16
print(f"Modelo completo (fp16):          ~{modelo_mb:,.0f} MB")
for carpeta in ["adaptador_sft", "adaptador_instrucciones"]:
    mb = tamano_mb(carpeta)
    if mb:
        print(f"{carpeta:32s} ~{mb:,.0f} MB  ({modelo_mb / mb:.0f} veces más liviano)")
    else:
        print(f"{carpeta:32s} (ejecuta la parte correspondiente para generarlo)")

**Para la discusión:** con un mismo modelo base se pueden tener muchos adaptadores livianos, uno por tarea (extraer inscripciones, seguir instrucciones, un tutor con personalidad), y cambiar entre ellos sin duplicar el modelo completo.

---
## Resumen

| Concepto | Qué se reutiliza | Qué se entrena | Resultado en la demo |
| --- | --- | --- | --- |
| Transfer Learning | Pesos de BETO | Todo el modelo + cabeza de clasificación | Mucho mejor que entrenar desde cero con los mismos 40 ejemplos |
| Supervised Fine-Tuning | LLM instruct | Pares entrada → JSON | Formato consistente y campos correctos |
| Instruction Tuning | LLM base | Instrucciones variadas → respuestas | Responde y se detiene en vez de seguir escribiendo |
| PEFT (LoRA) | Todo el modelo congelado | Menos del 2 % de los parámetros | Adaptadores de pocos MB |